In [ ]:
# Penalidade de temperatura: desvio da faixa ideal (22°C)

  pen_temp = abs(temp - 22) * 0.25

  pen_temp = min(pen_temp, 4) # máximo de penalidade: 4 pontos



  # Penalidade de umidade: ideal entre 60-70%

  if 60 <= umidade <= 70:

    pen_umidade = 0

  elif umidade < 60:

    pen_umidade = (60 - umidade) * 0.05

  else:

    pen_umidade = (umidade - 70) * 0.04

  pen_umidade = min(pen_umidade, 3) # máximo: 3 pontos



  # Penalidade do IQA: normaliza 0-100 para 0-3 pontos de penalidade

  pen_iqa = (iqa / 100) * 3

  pen_iqa = min(pen_iqa, 3)



  nota = 10 - pen_temp - pen_umidade - pen_iqa

  return round(max(nota, 0), 2)





def classificar_iqa(iqa):

  """Classifica o IQA conforme faixas oficiais da CETESB (Brasil)."""

  match True:

    case _ if iqa <= 40:

      return "Boa", "Sem riscos à saúde"

    case _ if iqa <= 80:

      return "Moderada", "Risco leve para grupos sensíveis"

    case _ if iqa <= 120:

      return "Ruim", "Risco à saúde da população em geral"

    case _ if iqa <= 200:

      return "Muito Ruim", "Risco sério — evitar exposição prolongada"

    case _:

      return "Péssima", "Emergência de saúde pública"





def analisar_microclima():

  """Função principal: processa os dados dos três locais em dois horários."""



  # Lista de listas: [local, horário, temp, umidade, iqa]

  dados_locais = [

    ["Ermelino Matarazzo", "Manhã", 19, 78, 42],

    ["Ermelino Matarazzo", "Tarde", 29, 55, 68],

    ["São Miguel Paulista", "Manhã", 18, 82, 38],

    ["São Miguel Paulista", "Tarde", 31, 48, 80],

    ["Tatuapé",       "Manhã", 20, 72, 30],

    ["Tatuapé",       "Tarde", 28, 58, 55],

  ]



  rotulos = ["Local", "Horário", "Temp (°C)", "Umidade (%)", "IQA"]



  print("=" * 60)

  print("   ANÁLISE DE MICROCLIMA — ZONA LESTE DE SÃO PAULO")

  print("=" * 60)



  for registro in dados_locais:

    print(f"\n {registro[0]} | {registro[1]}")

    print("-" * 40)



    # Loop sobre as métricas do registro (índices 2 a 4)

    metricas = ["Temperatura", "Umidade", "IQA"]

    for i, metrica in enumerate(metricas):

      valor = registro[i + 2]

      print(f" {metrica}: {valor}")



    # Desempacota para clareza

    _, _, temp, umidade, iqa = registro



    # Classificação do IQA com match-case

    qualidade, risco = classificar_iqa(iqa)

    print(f" Qualidade do Ar: {qualidade} — {risco}")



    # Nota de conforto urbano

    nota = calcular_nota_conforto(temp, umidade, iqa)

    print(f" Nota de Conforto Urbano: {nota}/10")



  print("\n" + "=" * 60)

  print("Análise concluída.")





# Execução

analisar_microclima()



In [ ]:
def simular_evacuacao():

  """

  Simulador de evacuação em uma faculdade.

  O agente parte de uma sala no 1º andar e tenta chegar a uma das 3 saídas.

  Possui energia limitada e um inventário básico.

  """



  # Lista dos nós do trajeto (do mais distante à saída)

  nos = [

    "Sala de Aula 105",    # 0 — ponto de partida

    "Corredor do 1º Andar",  # 1

    "Topo da Escada A",    # 2

    "Base da Escada A",    # 3

    "Hall Térreo Central",   # 4

    "Bifurcação das Saídas",  # 5

    "SAÍDA PRINCIPAL",     # 6 — saída 1

  ]



  # Estados/obstáculos de cada nó

  estados = [

    "livre",          # Sala 105

    "fumaça leve",       # Corredor do 1º Andar

    "extintor disponível",   # Topo da Escada A

    "livre",          # Base da Escada A

    "porta trancada",     # Hall Térreo Central

    "livre",          # Bifurcação

    "SAÍDA LIVRE",       # Saída Principal

  ]



  # Saídas alternativas (caso o agente chegue à bifurcação)

  saidas_alternativas = ["Saída Lateral B", "Saída de Emergência C"]



  # Estado inicial do agente

  posicao = 0

  energia = 10    # cada movimento custa 1; obstáculos custam extra

  inventario = []   # o agente pode coletar itens ao longo do percurso

  chave_encontrada = False



  print("=" * 55)

  print("    SIMULADOR DE EVACUAÇÃO — FACULDADE")

  print("=" * 55)

  print(f"\n Agente inicia em: {nos[posicao]}")

  print(f" Energia inicial: {energia}")

  print(f" Inventário: {inventario if inventario else 'vazio'}")

  print()



  while posicao < len(nos) - 1 and energia > 0:

    proximo = posicao + 1

    no_atual = nos[posicao]

    obstaculo = estados[proximo]



    print(f" Tentando avançar para: {nos[proximo]}")

    print(f" Obstáculo detectado: '{obstaculo}'")



    # Avaliação do obstáculo com if aninhado

    if obstaculo == "livre" or obstaculo == "SAÍDA LIVRE":

      print("  Caminho livre! Avançando.")

      posicao = proximo

      energia -= 1



    elif obstaculo == "fumaça leve":

      print("  Fumaça detectada.")

      if "extintor" in inventario:

        print("  Usando extintor! Fumaça controlada.")

        inventario.remove("extintor")

        posicao = proximo

        energia -= 2

      else:

        print("  Sem extintor. Atravessando com cuidado (perda extra de energia).")

        posicao = proximo

        energia -= 3



    elif obstaculo == "extintor disponível":

      print("  Extintor encontrado! Adicionado ao inventário.")

      inventario.append("extintor")

      posicao = proximo

      energia -= 1



    elif obstaculo == "porta trancada":

      print("  Porta trancada!")

      if chave_encontrada or "chave" in inventario:

        print("  Chave no inventário. Destrancando porta.")

        posicao = proximo

        energia -= 1

      else:

        # Subcondição: verificar se há rota alternativa disponível

        print("  Sem chave! Verificando rotas alternativas...")

        if saidas_alternativas:

          alternativa = saidas_alternativas.pop(0)

          print(f" Redirecionando para: {alternativa}")

          print(f" {alternativa} — AGENTE EVACUADO COM SUCESSO!")

          energia -= 2

          print(f"\n Energia restante: {energia}")

          print("=" * 55)

          return

        else:

          print("  Nenhuma alternativa restante! Recuando...")

          posicao = max(0, posicao - 1)

          energia -= 2



    else:

      # Obstáculo desconhecido — tenta avançar mesmo assim

      print(f"  Obstáculo desconhecido. Tentando avançar com cautela.")

      posicao = proximo

      energia -= 2



    print(f"  Posição atual: {nos[posicao]}")

    print(f"  Energia restante: {energia}")

    print(f" Inventário: {inventario if inventario else 'vazio'}")

    print()



  # Verificação do resultado final

  print("=" * 55)

  if posicao == len(nos) - 1:

    print(" EVACUAÇÃO CONCLUÍDA! O agente chegou à SAÍDA PRINCIPAL.")

  elif energia <= 0:

    print(" FALHA NA EVACUAÇÃO. Agente sem energia.")

    print(f"  Última posição: {nos[posicao]}")

  print(f" Energia final: {energia}")

  print("=" * 55)





# Execução

simular_evacuacao()